In [14]:
# Cell 1: Setup, imports, and data load
#
# External validation against real PAN-OS firewall logs, re-aligned to follow
# the production scoring pipeline (dashboard/adaptrap_firewall_pipeline.py):
# a single frozen checkpoint (M5), three-tier ALLOW/ESCALATE_TO_ANALYST/BLOCK
# rules at the same percentiles, and the same quality metrics (flagging rate,
# anomaly score distribution, silhouette coefficient).
#
# - combined_logs_drop_type.csv : main dataset -> adapted + profiled + scored
#   by the model (simulates the "unlabeled" operational log the firewall sees).
# - combined_logs.csv           : same traffic, but includes the PAN "Type"/
#   "Action" columns -> used ONLY afterwards, as ground truth to score accuracy.

import sys
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report, silhouette_score,
)

sys.path.insert(0, '/Users/gabriellecabangcala/thesis/AdapTrap/models')
from adaptrap_pipeline import (
    build_ip_profiles,
    get_feature_cols,
    compute_conn_rate,
    generate_rules,
)

# Same two-threshold config as the production pipeline
ESCALATE_PERCENTILE = 75
BLOCK_PERCENTILE = 95

MAIN_PATH = '/Users/gabriellecabangcala/thesis/AdapTrap/test/combined_logs_drop_type.csv'
TRUTH_PATH = '/Users/gabriellecabangcala/thesis/AdapTrap/test/combined_logs.csv'

df_main = pd.read_csv(MAIN_PATH, low_memory=False)
df_truth = pd.read_csv(TRUTH_PATH, low_memory=False)

print(f"Main (unlabeled) dataset: {df_main.shape}")
print(f"Truth (labeled) dataset:  {df_truth.shape}")


Main (unlabeled) dataset: (66018, 148)
Truth (labeled) dataset:  (66018, 149)


In [15]:
# Cell 2: Adapt the PAN-OS schema to the AdapTrap packet-level schema
#
# adaptrap_pipeline.build_ip_profiles()/compute_conn_rate() expect one row per
# packet with: source_ip, time_sec, protocol, length, dst_port, has_syn,
# has_ack, has_fin, has_rst, has_psh. PAN traffic logs are per-session, not
# per-packet, and their Flags column is a session bitmask (not TCP flag
# letters), so the TCP-flag ratios can't be recovered and are left at 0 -
# consistent with the adaptation used for the earlier PAN external-validation
# pass (test/run_validation.py).

def adapt_schema(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns={
        "Source address": "source_ip",
        "Destination Port": "dst_port",
        "Bytes": "length",
        "Application": "protocol",
    }).copy()

    df["time_sec"] = pd.to_datetime(df["Time Logged"], errors="coerce").astype("int64") // 10**9
    df = df.dropna(subset=["source_ip", "time_sec"])

    df["dst_port"] = pd.to_numeric(df["dst_port"], errors="coerce").fillna(0)
    df["length"] = pd.to_numeric(df["length"], errors="coerce").fillna(0)
    df["protocol"] = df["protocol"].fillna("other")

    for flag_col in ["has_syn", "has_ack", "has_fin", "has_rst", "has_psh"]:
        df[flag_col] = 0

    return df[["source_ip", "time_sec", "protocol", "length", "dst_port",
               "has_syn", "has_ack", "has_fin", "has_rst", "has_psh"]]

adapted_df = adapt_schema(df_main)
adapted_df = compute_conn_rate(adapted_df)
print(adapted_df.shape)
adapted_df.head()


(66018, 11)


,source_ip,time_sec,protocol,length,dst_port,has_syn,has_ack,has_fin,has_rst,has_psh,conn_rate
0,1.178.136.86,1790238,web-browsing,47316.0,443,0,0,0,0,0,1
1,1.187.66.21,1790214,web-browsing,27178.0,443,0,0,0,0,0,1
2,1.38.98.12,1790198,ssl,6285.0,443,0,0,0,0,0,1
3,1.4.192.61,1790152,web-browsing,3157.0,443,0,0,0,0,0,1
4,1.52.1.99,1790201,web-browsing,106947.0,443,0,0,0,0,0,1


In [16]:
# Cell 3: Build attacker IP profiles (same aggregation as training)

profiles = build_ip_profiles(adapted_df)
print(f"Behavioral IP profiles: {profiles.shape}")
profiles.head()


Behavioral IP profiles: (8938, 24)


,source_ip,total_packets,unique_dst_ports,protocol_diversity,avg_length,std_length,max_length,syn_ratio,ack_ratio,fin_ratio,...,session_duration,packets_per_second,pct_tcp,pct_vnc,pct_tlsv12,pct_sshv2,pct_dns,pct_tds,pct_telnet,pct_http
0,1.178.136.86,1.0,1.0,1.0,47316.0,0.0,47316.0,0.0,0.0,0.0,...,0.000001,1000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.187.66.21,1.0,1.0,1.0,27178.0,0.0,27178.0,0.0,0.0,0.0,...,0.000001,1000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.38.98.12,1.0,1.0,1.0,6285.0,0.0,6285.0,0.0,0.0,0.0,...,0.000001,1000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.4.192.61,1.0,1.0,1.0,3157.0,0.0,3157.0,0.0,0.0,0.0,...,0.000001,1000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.52.1.99,1.0,1.0,1.0,106947.0,0.0,106947.0,0.0,0.0,0.0,...,0.000001,1000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
# Cell 4: Load the published M5 checkpoint (final/frozen, same as the
# production dashboard pipeline) and score every attacker IP profile from
# the external PAN dataset with the same three-tier rule engine.

MODEL_DIR = '/Users/gabriellecabangcala/thesis/AdapTrap/saved_models'
model = joblib.load(f'{MODEL_DIR}/m5_IF_model')
scaler = joblib.load(f'{MODEL_DIR}/m5_scaler')
FEATURE_COLUMNS = joblib.load(f'{MODEL_DIR}/m5_feature_cols')

# Align profile columns to the exact feature order the model was trained on;
# any pct_<protocol> column absent from this dataset's traffic mix is 0.
for col in FEATURE_COLUMNS:
    if col not in profiles.columns:
        profiles[col] = 0

X_scaled = scaler.transform(profiles[FEATURE_COLUMNS])

# No port lookup: enforcement blocks the full IP, not a specific port -
# matches the production pipeline exactly.
rules_df = generate_rules(
    model, X_scaled, profiles, {}, label="External Validation - PAN Firewall Logs (M5)",
    escalate_percentile=ESCALATE_PERCENTILE, block_percentile=BLOCK_PERCENTILE,
)



--- External Validation - PAN Firewall Logs (M5) ---
         source_ip  port  anomaly_score action
0  112.211.120.226  None       0.096108  BLOCK
1    103.151.28.82  None       0.092162  BLOCK
2   138.84.137.210  None       0.092090  BLOCK
3    143.44.236.64  None       0.092090  BLOCK
4   180.190.117.27  None       0.090367  BLOCK
5   112.207.208.91  None       0.089666  BLOCK
6    136.158.8.100  None       0.089571  BLOCK
7   112.200.33.214  None       0.089264  BLOCK
8   175.176.41.181  None       0.089146  BLOCK
9  112.210.227.115  None       0.088807  BLOCK

ALLOW: 4634, ESCALATE_TO_ANALYST: 3677, BLOCK: 627


/opt/homebrew/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreeRegressor from version 1.6.1 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/homebrew/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator IsolationForest from version 1.6.1 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/homebrew/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.9.1. This might lead to bre

In [18]:
# Cell 5: Model-quality metrics, same definitions as the production pipeline
#
# silhouette_coefficient is computed from the model's own contamination-based
# decision (model.predict), matching Section 3.4.1 - NOT the percentile
# escalation threshold used by generate_rules. These are intentionally two
# different things: this reports model quality, generate_rules decides the
# operational firewall action.

def flagging_rate(actions: np.ndarray) -> float:
    return float(np.mean(actions != "ALLOW") * 100)


def anomaly_score_distribution(scores: np.ndarray) -> dict:
    return {
        "mean": float(np.mean(scores)),
        "std": float(np.std(scores)),
        "min": float(np.min(scores)),
        "max": float(np.max(scores)),
    }


def silhouette_coefficient(model, X: np.ndarray):
    flags = (model.predict(X) == -1).astype(int)
    if len(set(flags)) < 2:
        return None
    return float(silhouette_score(X, flags))


scores = rules_df["anomaly_score"].to_numpy()
actions = rules_df["action"].to_numpy()

metrics = {
    "flagging_rate_pct": round(flagging_rate(actions), 2),
    "anomaly_score_distribution": anomaly_score_distribution(scores),
    "silhouette_coefficient": silhouette_coefficient(model, X_scaled),
    "action_breakdown": {
        a: int((actions == a).sum()) for a in ["ALLOW", "ESCALATE_TO_ANALYST", "BLOCK"]
    },
}
metrics


{'flagging_rate_pct': 48.15,
 'anomaly_score_distribution': {'mean': -0.0033438850079745455,
  'std': 0.03399698149117343,
  'min': -0.11835751050490045,
  'max': 0.09610839031138052},
 'silhouette_coefficient': 0.09001030678602433,
 'action_breakdown': {'ALLOW': 4634,
  'ESCALATE_TO_ANALYST': 3677,
  'BLOCK': 627}}

In [23]:
# Cell 6: Side-by-side model actions vs PAN ground-truth labels
#
# Ground truth uses the labeled dataset's two labels: ALLOW and THREAT.
# The model output remains three-tiered: ALLOW, ESCALATE_TO_ANALYST, BLOCK.

truth_rows = df_truth[["Source address", "Type"]].copy()
truth_rows["Source address"] = truth_rows["Source address"].astype(str)
truth_rows["Type"] = truth_rows["Type"].fillna("").str.upper()
truth_rows["ground_truth_label"] = np.where(
    truth_rows["Type"].eq("THREAT"), "THREAT", "ALLOW"
)

truth_by_ip = (
    truth_rows.sort_values("ground_truth_label")
    .drop_duplicates("Source address", keep="last")
    [["Source address", "ground_truth_label"]]
    .rename(columns={"Source address": "source_ip"})
)

comparison_df = rules_df[["source_ip", "anomaly_score", "action"]].merge(
    truth_by_ip, on="source_ip", how="left"
)
comparison_df["ground_truth_label"] = comparison_df["ground_truth_label"].fillna("ALLOW")
comparison_df["model_binary_label"] = np.where(
    comparison_df["action"].eq("ALLOW"), "ALLOW", "THREAT"
)
comparison_df["match"] = (
    comparison_df["model_binary_label"] == comparison_df["ground_truth_label"]
)
comparison_df = comparison_df.rename(columns={
    "source_ip": "Source IP",
    "anomaly_score": "Anomaly Score",
    "action": "Model Action",
    "ground_truth_label": "Ground Truth Label",
    "model_binary_label": "Model Binary Label",
    "match": "Match",
})

print("Side-by-side comparison:")
display(comparison_df.sort_values("Anomaly Score", ascending=False).head(30))

comparison_df.to_csv(
    "/Users/gabriellecabangcala/thesis/AdapTrap/test/validation_comparison.csv",
    index=False,
)

binary_labels = ["ALLOW", "THREAT"]
print(f"\nIPs evaluated: {len(comparison_df)}")
print(f"Binary agreement: {comparison_df['Match'].mean():.2%}")
print("\nConfusion Matrix [rows=ground truth, cols=model] (ALLOW, THREAT):")
print(confusion_matrix(
    comparison_df["Ground Truth Label"],
    comparison_df["Model Binary Label"],
    labels=binary_labels,
))
print("\nClassification Report:")
print(classification_report(
    comparison_df["Ground Truth Label"],
    comparison_df["Model Binary Label"],
    labels=binary_labels,
    zero_division=0,
))

Side-by-side comparison:


,Source IP,Anomaly Score,Model Action,Ground Truth Label,Model Binary Label,Match
0,112.211.120.226,0.096108,BLOCK,ALLOW,THREAT,False
1,103.151.28.82,0.092162,BLOCK,ALLOW,THREAT,False
2,138.84.137.210,0.092090,BLOCK,ALLOW,THREAT,False
3,143.44.236.64,0.092090,BLOCK,ALLOW,THREAT,False
4,180.190.117.27,0.090367,BLOCK,ALLOW,THREAT,False
5,112.207.208.91,0.089666,BLOCK,ALLOW,THREAT,False
6,136.158.8.100,0.089571,BLOCK,ALLOW,THREAT,False
7,112.200.33.214,0.089264,BLOCK,ALLOW,THREAT,False
8,175.176.41.181,0.089146,BLOCK,ALLOW,THREAT,False
9,112.210.227.115,0.088807,BLOCK,ALLOW,THREAT,False



IPs evaluated: 8938
Binary agreement: 51.83%

Confusion Matrix [rows=ground truth, cols=model] (ALLOW, THREAT):
[[4632 4303]
 [   2    1]]

Classification Report:
              precision    recall  f1-score   support

       ALLOW       1.00      0.52      0.68      8935
      THREAT       0.00      0.33      0.00         3

    accuracy                           0.52      8938
   macro avg       0.50      0.43      0.34      8938
weighted avg       1.00      0.52      0.68      8938

